# 11 · Pretrained vs Fine-tuned SC-FEGAN 비교

**목적**: 동일 입력에 대해 SC-FEGAN 두 버전을 inference하여 fine-tuning 효과 시각 검증.

**전제 조건**:
- 노트북 10에서 fine-tuning 완료 → `scfegan_finetuned.ckpt` 존재
- 또는 fallback으로 pretrained만 사용

**산출물**: `outputs/pretrained_vs_finetuned.png` — 시술 3종 × 2 ckpt = 6장 비교

In [ ]:
import os, sys
IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/cv-final'
    if not os.path.exists(REPO):
        from google.colab import userdata
        token = userdata.get('GH_TOKEN')
        !git clone https://{token}@github.com/keonU206/cv-final.git $REPO
    %cd $REPO
    os.environ['TF_USE_LEGACY_KERAS'] = '1'
    !pip install -q mediapipe==0.10.21 opencv-python pyyaml
print(f'CWD: {os.getcwd()}')

## 1. 입력 사진 + 랜드마크

In [ ]:
import numpy as np, cv2, matplotlib.pyplot as plt
from pathlib import Path
SIZE = 512
OUTPUT_DIR = Path('outputs/comparison')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if IS_COLAB:
    from google.colab import files
    print('비교용 사진 1장 업로드:')
    uploaded = files.upload()
    IMG_PATH = list(uploaded.keys())[0]
else:
    IMG_PATH = 'demo_face.jpg'

image_rgb = cv2.cvtColor(cv2.imread(IMG_PATH), cv2.COLOR_BGR2RGB)
image_rgb = cv2.resize(image_rgb, (SIZE, SIZE))

import mediapipe as mp
with mp.solutions.face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1, refine_landmarks=True) as fm:
    res = fm.process(image_rgb)
lms = res.multi_face_landmarks[0].landmark
landmarks = np.array([[lm.x * SIZE, lm.y * SIZE] for lm in lms], dtype=np.int32)
print(f'Landmarks: {landmarks.shape}')

## 2. 두 ckpt 비교 inference

In [ ]:
from project.input_generator import compose_scfegan_input, to_scfegan_tensor
from project.scfegan_finetune.train_finetune import setup_tf_compat

setup_tf_compat()
import tensorflow.compat.v1 as tf

PRETRAINED = '/content/drive/MyDrive/SC-FEGAN-ckpt/SC-FEGAN.ckpt'
FINETUNED = '/content/drive/MyDrive/cv-final-ckpts/scfegan_finetuned/scfegan_finetuned.ckpt'

have_pretrained = Path(PRETRAINED + '.index').exists()
have_finetuned = Path(FINETUNED + '.index').exists()
print(f'pretrained available: {have_pretrained}')
print(f'fine-tuned available: {have_finetuned}')

if not have_finetuned:
    print('⚠ fine-tuned ckpt 없음 — pretrained만 시각화')

In [ ]:
def run_scfegan(ckpt_path, composed):
    """SC-FEGAN graph 빌드 + inference.
    
    Phase 7-C1 fine-tuning과 동일한 graph 사용.
    실패 시 incomplete_image 반환.
    """
    sys.path.insert(0, '/content/SC-FEGAN')
    try:
        from src.model import Model as SCFEGANModel
        tf.reset_default_graph()
        model = SCFEGANModel(size=SIZE, mode='test')
        saver = tf.train.Saver()
        with tf.Session() as sess:
            saver.restore(sess, ckpt_path)
            tensor = to_scfegan_tensor(
                composed, normalize=True, channels_first=False, batch_dim=True,
            )
            # SC-FEGAN 원본 input placeholder 이름은 model 코드 확인 필요
            # 일반적으로 model.input_ph 또는 model.test_input
            output = sess.run(
                model.output if hasattr(model, 'output') else model.gen_out,
                feed_dict={model.input_ph: tensor},
            )
        return ((output[0] + 1) * 127.5).clip(0, 255).astype(np.uint8)
    except Exception as e:
        print(f'  inference 실패 ({Path(ckpt_path).name}): {e}')
        return composed['incomplete_image']

results = {}
for proc_id in ['nose_tip', 'double_eyelid', 'jaw_v_line']:
    print(f'\n{proc_id} 처리...')
    composed = compose_scfegan_input(image_rgb, landmarks, proc_id, size=SIZE, intensity=0.6, seed=42)
    res_pre = run_scfegan(PRETRAINED, composed) if have_pretrained else composed['incomplete_image']
    res_ft = run_scfegan(FINETUNED, composed) if have_finetuned else res_pre
    results[proc_id] = {'pretrained': res_pre, 'finetuned': res_ft, 'composed': composed}
    print(f'  ✅ {proc_id} 완료')

## 3. 비교 시각화 — 3 시술 × 3 컬럼 (Before / Pretrained / Fine-tuned)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 13))
for i, (proc_id, r) in enumerate(results.items()):
    axes[i, 0].imshow(image_rgb)
    axes[i, 0].set_title(f'{proc_id} · Before')
    axes[i, 1].imshow(r['pretrained'])
    axes[i, 1].set_title('Pretrained SC-FEGAN')
    axes[i, 2].imshow(r['finetuned'])
    axes[i, 2].set_title('Fine-tuned SC-FEGAN ⭐')
    for ax in axes[i]:
        ax.axis('off')
plt.tight_layout()
save_path = OUTPUT_DIR / 'pretrained_vs_finetuned.png'
plt.savefig(save_path, dpi=120, bbox_inches='tight')
print(f'✅ 저장: {save_path}')
plt.show()

## 4. 정량 평가 — SSIM, PSNR (선택)

Fine-tuned가 더 자연스러운지 객관 지표로 측정. GT가 없어서 직접 비교는 어려우니, **mask 영역의 색상 연속성**을 측정.

In [ ]:
from skimage.metrics import structural_similarity as ssim

print(f'{"시술":<15} {"SSIM (pre vs ft)":<20} {"평균 색상 연속성 향상":<25}')
print('-' * 65)
for proc_id, r in results.items():
    # 두 결과 사이의 SSIM (얼마나 다른지)
    s = ssim(r['pretrained'], r['finetuned'], channel_axis=2, data_range=255)
    # mask 경계에서의 색상 그래디언트 (낮을수록 자연스러움)
    mask = r['composed']['mask'][..., 0] > 0
    edges = cv2.Canny(mask.astype(np.uint8) * 255, 100, 200)
    grad_pre = cv2.Laplacian(cv2.cvtColor(r['pretrained'], cv2.COLOR_RGB2GRAY), cv2.CV_64F)
    grad_ft = cv2.Laplacian(cv2.cvtColor(r['finetuned'], cv2.COLOR_RGB2GRAY), cv2.CV_64F)
    edge_pixels = edges > 0
    cont_pre = np.abs(grad_pre[edge_pixels]).mean() if edge_pixels.any() else 0
    cont_ft = np.abs(grad_ft[edge_pixels]).mean() if edge_pixels.any() else 0
    delta = cont_pre - cont_ft  # 양수 = fine-tuned가 더 부드러움
    print(f'{proc_id:<15} {s:<20.4f} {delta:+.4f}')

## 5. 발표용 단일 이미지 저장

In [ ]:
# Drive에 백업
if IS_COLAB:
    DRIVE_OUT = '/content/drive/MyDrive/cv-final-ckpts/samples'
    Path(DRIVE_OUT).mkdir(parents=True, exist_ok=True)
    !cp {save_path} {DRIVE_OUT}/
    print(f'✅ Drive 백업: {DRIVE_OUT}/pretrained_vs_finetuned.png')

print('\nPhase 7-C1 비교 검증 완료. 발표 슬라이드에 추가 가능.')